# PitchOps — Football MLOps trên Colab

Notebook của **Nguyễn Hồng Quang**. Chạy CPU, không cần GPU. **USE_DRIVE=True** tự lưu dữ liệu, model, ảnh và dashboard vào Drive. Notebook chạy cài đặt, kiểm thử và huấn luyện tương tác tại Colab; CI và job hằng tuần chạy độc lập trên GitHub Actions.

**CLB:** 5 giải lớn, bàn/thắng–hòa–thua, góc/cú sút thử nghiệm, Monte Carlo quốc nội, logo/ảnh và metrics.

**Event Lab:** World Cup 2022 replay lịch sử riêng cho cầu thủ, ném biên/control proxy, PI, MOTM-index, heatmap Bayesian, XI theo lịch biết trước và giải cúp. Không phải live coverage toàn cầu hoặc official ratings/tracking. GitHub Actions cào dữ liệu và train hằng tuần; Colab nhận model mới vào Drive khi chạy đồng bộ.

Dữ liệu football-data.co.uk chỉ dùng cá nhân/học thuật theo điều khoản nguồn, tải nhẹ tối đa hai lần/tuần. Giữ cache/ledger qua các phiên. [Code & hướng dẫn](https://github.com/HongQuang26/pitchops).


## 1. Lấy code và cài thư viện

Clone lần đầu, lần sau tái dùng checkout. Không tự ghi đè các thay đổi bạn đang sửa. In commit để đối chiếu code chạy thật. Môi trường chính xác được ghi lại trong artifacts sau khi chạy.

In [ ]:
from pathlib import Path
import os, subprocess, sys, importlib
REPO = Path('/content/pitchops')
if not (REPO / '.git').exists():
    subprocess.run(['git','clone','https://github.com/HongQuang26/pitchops.git',str(REPO)], check=True)
else:
    changes = subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()
    if not changes:
        subprocess.run(['git','pull','--ff-only'],cwd=REPO,check=True)
    else:
        print('Keeping your uncommitted code edits; git pull skipped.')
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-q','-c','requirements/runtime.txt','-e','.[dev,automation]'],check=True)
if str(REPO/'src') not in sys.path:
    sys.path.insert(0,str(REPO/'src'))
importlib.invalidate_caches()
if 'pitchops' in sys.modules and getattr(sys.modules['pitchops'],'__file__',None) is None:
    del sys.modules['pitchops']
for name in ['pitchops.data','pitchops.storage','pitchops.media','pitchops.dashboard','pitchops.pipeline','pitchops.event_data','pitchops.advanced_models','pitchops.forecast_store','pitchops.cups','pitchops.research','pitchops.artifact_sync','pitchops.weekly']:
    if name in sys.modules:
        importlib.reload(sys.modules[name])
print('Code commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('Ready:',REPO)


## 2. Nơi giữ cache và checkpoint

`USE_DRIVE=True` là mặc định. Xác nhận Google khi được hỏi. Cache, model, ảnh và dashboard giữ tại `MyDrive/PitchOps/workspace`. Chuyển workspace cũ có kiểm tra hash. Không reset ledger để vượt quota hoặc chạy nhiều phiên ghi cùng workspace.


In [ ]:
USE_DRIVE = True
from pitchops.storage import migrate_workspace, verify_persistence
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
WORKSPACE = Path('/content/drive/MyDrive/PitchOps/workspace') if USE_DRIVE else Path('/content/pitchops_workspace')
WORKSPACE.mkdir(parents=True, exist_ok=True)
if USE_DRIVE:
    print(migrate_workspace('/content/pitchops_workspace',WORKSPACE))
print(verify_persistence(WORKSPACE))
print('Persistent across runtimes:',USE_DRIVE)

# Receive weekly encrypted model/cache from GitHub Actions when available.
AUTO_SYNC_ACTIONS = True
if AUTO_SYNC_ACTIONS:
    from pitchops.artifact_sync import sync_and_manage
    print(sync_and_manage(WORKSPACE))


## 3. Kiểm thử các hợp đồng quan trọng

Kiểm tra rò rỉ dữ liệu tương lai, xử lý trận cùng ngày, tổng xác suất, dữ liệu trùng/âm, coverage mùa giải, nhãn cầu thủ và cache budget. GitHub Actions cũng chạy CI và pipeline hằng tuần với cùng constraint môi trường.

In [ ]:
test_result = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO, capture_output=True, text=True)
print(test_result.stdout)
print(test_result.stderr)
test_result.check_returncode()


## 4. Pipeline và đồng bộ logo miniface

`RETRAIN=True` lấy dữ liệu hợp lệ từ cache/nguồn rồi huấn luyện và đánh giá như trước. Chỉ muốn cập nhật ảnh/giao diện trên model hiện có: đổi `RETRAIN=False`. Cập nhật ảnh không sửa snapshot huấn luyện; tạo một dashboard view có phiên bản riêng.

`SYNC_MEDIA=True` thêm logo 5 giải, logo đội từ nguồn metadata và logo dự phòng, miniface và hồ sơ thật của các đội giải được chọn. Free API giới hạn số đội/cầu thủ trả về; dashboard ghi rõ danh sách một phần. Bản đầu đồng bộ tối đa 50 đội có ID khớp, rate limit tối đa 30 requests/phút, cache 7 ngày; lần đầu có thể mất vài phút. Logo và 48 ảnh cầu thủ đầu tiên được cache/nhúng để giảm phụ thuộc mạng; các ảnh khác tải khi cần. Không dùng danh mục ảnh làm dữ liệu huấn luyện hoặc khẳng định đội hình thi đấu.

Không cần API key trả phí. Nếu bạn có key riêng, tự thêm `SPORTSDB_API_KEY` trong Colab Secrets. Không chia sẻ key trong code/chat. Artwork vẫn thuộc quyền của chủ sở hữu; mục hồ sơ có nguồn và credit.


In [ ]:
from pitchops.pipeline import run
from pitchops.storage import latest_dashboard
from pitchops.media import enrich_dashboard
RETRAIN = False
SYNC_MEDIA = True
SEASONS = ['2526','2627']
LEAGUES = ['E0','SP1','D1','I1','F1']
AS_OF = None
LOCAL_FILES = None
if RETRAIN or not (WORKSPACE/'artifacts'/'latest.json').exists():
    payload,run_dir = run(WORKSPACE,seasons=SEASONS,leagues=LEAGUES,as_of=AS_OF,simulations=5000,local_files=LOCAL_FILES)
else:
    payload,run_dir = latest_dashboard(WORKSPACE)
media_key = None
if SYNC_MEDIA:
    try:
        from google.colab import userdata
        media_key = userdata.get('SPORTSDB_API_KEY')
    except (userdata.SecretNotFoundError,userdata.NotebookAccessError):
        pass  # Default public development key; no privileged access required.
    payload,run_dir = enrich_dashboard(WORKSPACE,api_key=media_key,max_rosters=50,embed_images=True)
    del media_key
print('Dashboard:',run_dir/'dashboard.html')
print('Saved to Drive:',USE_DRIVE)


## 5. Dashboard phong cách sân cỏ

Logo xuất hiện ở bộ chọn giải, trận đấu và bảng đua. Mục **Cầu thủ** có miniface thật, tìm theo tên, lọc đội/vị trí, hồ sơ nguồn và danh sách theo dõi. Bộ lọc và theo dõi chạy trong trình duyệt; việc lưu favorites có thể bị chặn trong iframe sandbox. Không cần tạo tài khoản hoặc public tunnel.

Chạy lại ô này để áp dụng giao diện mới từ snapshot Drive, không cào hoặc train lại. View được tái dùng khi asset UI không thay đổi. Đổi `PREVIEW_WIDTH='390px'` để xem bố cục di động. HTML có snapshot và các ảnh đã nhúng; ảnh chưa cache cần mạng, thiếu ảnh sẽ hiện initials.


In [ ]:
from IPython.display import HTML,display,clear_output
from pitchops.dashboard import refresh_dashboard
import html,warnings
payload,run_dir = refresh_dashboard(WORKSPACE)
page = (run_dir/'dashboard.html').read_text(encoding='utf-8')
PREVIEW_WIDTH = '100%'
clear_output(wait=True)
with warnings.catch_warnings():
    warnings.filterwarnings('ignore',message='Consider using IPython.display.IFrame instead')
    display(HTML('<iframe title="PitchOps dashboard" sandbox="allow-scripts allow-downloads" style="width:'+PREVIEW_WIDTH+';height:1150px;border:0;border-radius:16px" srcdoc="'+html.escape(page,quote=True)+'"></iframe>'))


## 6. Metrics nguồn gốc và độ phủ ảnh

Trọng số/model card vẫn nằm trong `artifacts/runs/<run_id>`. View có logo và miniface nằm trong `artifacts/views/<view_id>` để không ghi đè snapshot huấn luyện. `latest.json` và `latest_view.json` trỏ đến bản mới nhất. File ảnh và metadata được cache trong `data/media` của Drive. Không có token/API key trong snapshot.


In [ ]:
import json
print(json.dumps({'meta':payload['meta'],'metrics':payload['metrics'],'quality':payload['quality'],'media_coverage':payload.get('media',{}).get('coverage',{})},ensure_ascii=False,indent=2))


## 7. MLflow tùy chọn

Bật `ENABLE_MLFLOW=True` để ghi experiment tại workspace. Không cần server MLflow hoặc tài khoản. Đây là run tracking; chưa có production registry/promotion tự động.

In [ ]:
ENABLE_MLFLOW = False
if ENABLE_MLFLOW:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[tracking]'], check=True)
    from pitchops.pipeline import track_mlflow
    track_mlflow(payload, run_dir)
    print('MLflow run logged')
else:
    print('Optional MLflow skipped; core JSON metrics and lineage are already saved.')


## 8. Cầu thủ và heatmap tùy chọn

Scorer projection cần CSV thật: `player_id,player,team,date,minutes,goals`. Đặt ngày dự báo và số phút dự kiến trước trận; không dùng số phút thực tế của trận mục tiêu. XI lịch sử cần thêm `rating,position` (GK/DF/MF/FW). Các ví dụ dưới đây bị tắt đến khi bạn cung cấp dữ liệu, không tạo cầu thủ giả.

In [ ]:
PLAYER_CSV = None
if PLAYER_CSV:
    import pandas as pd
    from pitchops.players import scorer_projection, select_form_xi
    player_logs = pd.read_csv(PLAYER_CSV)
    EXPECTED_MINUTES = {}  # player_id thật -> số phút dự kiến
    print(scorer_projection(player_logs, payload['meta']['forecast_date'], EXPECTED_MINUTES))
    # display(pd.DataFrame(select_form_xi(player_logs, payload['meta']['forecast_date'], period='week')))
else:
    print('Player models need verified player match logs; no synthetic player outputs.')
# StatsBomb adapter, chỉ bật sau khi đọc license và dùng IDs từ competitions/matches:
# from pitchops.events import StatsBomb
# sb = StatsBomb(WORKSPACE / 'data' / 'statsbomb')
# display(sb.competitions())
# locations = sb.player_locations(match_id=..., player_id=...)
# Đây là tọa độ sự kiện quá khứ, không phải tracking/predicted heatmap.


## 9. Tải dashboard hoặc backup

Bật cờ bên dưới rồi chạy. Backup gồm cache/ledger và artifacts, không phải secret. Khôi phục cùng cache trước khi tải tự động ở runtime mới. Không đẩy backup này vào repo public. Chỉ chia sẻ output nếu quyền dữ liệu của bạn cho phép.

In [ ]:
DOWNLOAD_DASHBOARD = False
DOWNLOAD_BACKUP = False
if DOWNLOAD_DASHBOARD or DOWNLOAD_BACKUP:
    from google.colab import files
if DOWNLOAD_DASHBOARD:
    from pitchops.storage import latest_dashboard
    _, current_view = latest_dashboard(WORKSPACE)
    files.download(str(current_view / 'dashboard.html'))
if DOWNLOAD_BACKUP:
    import shutil
    backup = shutil.make_archive('/content/PitchOps_private_backup', 'zip', WORKSPACE)
    files.download(backup)


## 10. Sửa code và commit/push từ Colab

Sửa file trong `/content/pitchops/src` hoặc dùng `%%writefile`. Muốn push, thêm `GITHUB_TOKEN` vào Colab Secrets (fine-grained, chỉ repo này, Contents write), tự cho phép notebook truy cập, rồi đặt `PUSH_CODE=True`. Token không in ra, không nằm trong Git URL hoặc file. Chỉ stage danh sách file code; không stage dataset/model/output. Ô này tắt mặc định.

In [ ]:
PUSH_CODE = False
if PUSH_CODE:
    from google.colab import userdata
    import tempfile
    token = userdata.get('GITHUB_TOKEN')
    env = os.environ.copy()
    env['PITCHOPS_GH_TOKEN'] = token
    env['GIT_TERMINAL_PROMPT'] = '0'
    with tempfile.TemporaryDirectory() as temp:
        askpass = Path(temp) / 'askpass.py'
        askpass.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["PITCHOPS_GH_TOKEN"])\n')
        askpass.chmod(0o700)
        env['GIT_ASKPASS'] = str(askpass)
        subprocess.run(['git','config','user.name','Nguyen Hong Quang'], check=True)
        subprocess.run(['git','config','user.email','144700005+HongQuang26@users.noreply.github.com'], check=True)
        subprocess.run(['git','add','src','tests','docs','notebooks','README.md','LICENSE','pyproject.toml','.gitignore'], check=True)
        print(subprocess.check_output(['git','diff','--cached','--stat'], text=True))
        if subprocess.run(['git','diff','--cached','--quiet']).returncode:
            subprocess.run(['git','commit','-m','Update PitchOps from Colab'], check=True)
            subprocess.run(['git','push','origin','main'], env=env, check=True)
        else:
            print('No staged code changes.')
    env.pop('PITCHOPS_GH_TOKEN', None)
    del token
else:
    print('GitHub write skipped. Use PUSH_CODE=True only for intentional code updates.')


## Tuần sau và các bước tiếp

Mở lại notebook, chạy setup và nối Drive; `AUTO_SYNC_ACTIONS=True` nhận model/cache mới từ job GitHub. Giữ `RETRAIN=False` nếu chỉ muốn xem. Lịch chính thứ Sáu 14:23 giờ Việt Nam; kiểm tra trạng thái ở Actions. Khóa riêng ở `.keys` phải được giữ trên Drive.


## 11. Event Research: cầu thủ, heatmap và giải cúp

Bộ nghiên cứu dùng **World Cup 2022**, cutoff `2022-12-15`. Đây là replay lịch sử, tách với dự báo CLB hiện tại. Lần đầu tải có giới hạn 64 file sự kiện; các lần sau dùng cache Drive. `TRAIN_EVENT_RESEARCH=False` mở lại kết quả đã lưu. Bật `True` khi muốn huấn luyện lại.


In [ ]:
# Event Research: historical World Cup 2022 replay; model and cache auto-save to Drive
import importlib, sys
for name in ['pitchops.event_data','pitchops.advanced_models','pitchops.forecast_store','pitchops.cups','pitchops.research']:
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from pitchops.storage import latest_dashboard, verify_persistence
from pitchops.research import build_research
TRAIN_EVENT_RESEARCH = False
payload, RUN_DIR = latest_dashboard(WORKSPACE)
if TRAIN_EVENT_RESEARCH or payload.get('research', {}).get('report', {}).get('scorer') is None:
    payload, RUN_DIR = build_research(WORKSPACE, as_of='2022-12-15', simulations=3000)
else:
    print('Current event models restored:', payload['research']['run_id'])
print(verify_persistence(WORKSPACE))
print('Dashboard:', RUN_DIR / 'dashboard.html')

In [ ]:
# Verify the saved research snapshot and serialized models without retraining
import json, joblib, numpy as np
from pitchops.storage import latest_dashboard, migrate_workspace
from pitchops.advanced_models import predict_team
payload, RUN_DIR = latest_dashboard(WORKSPACE)
r = payload['research']
checks = {}
checks['all_matchup_probabilities'] = all(abs(f['home_win']+f['draw']+f['away_win']-1)<1e-9 for f in r['forecasts'].values())
checks['control_mass'] = all(abs(sum(f['control'])-1)<1e-9 for f in r['forecasts'].values())
checks['player_goal_share_reserve'] = all(abs(sum(p['goal_share'] for p in players)-.9)<1e-9 for players in r['team_players'].values())
checks['spatial_mass'] = all(abs(np.array(m['predicted']).sum()-1)<1e-6 for m in r['spatial'].values())
checks['cup_title_mass'] = abs(sum(t['title'] for t in r['cup']['table'])-1)<1e-9
checks['known_finalists'] = {t['team'] for t in r['cup']['table'] if t['title']>0} == {'Argentina','France'}
checks['complete_xi'] = all(len(xi['players'])==11 and len({p['id'] for p in xi['players']})==11 for xi in r['best_xi'].values())
checks['substitute_position'] = next(p for p in r['team_players']['Argentina'] if p['name']=='Lisandro Martínez')['position']=='DF'
checks['replay_not_live'] = r['report']['forecast_audit']['live']['n']==0 and r['report']['forecast_audit']['replay']['n']==2
model_dir = WORKSPACE/'artifacts'/'event_research'/'runs'/r['run_id']
checks['complete_artifacts'] = all((model_dir/name).exists() for name in ['models.joblib','source_manifest.json','model_card.json','requirements-lock.txt','metrics.json','player_match_logs.csv'])
models = joblib.load(model_dir/'models.joblib')
restored = predict_team(models['team_models'],models['team_state'],'Argentina','France','SB43_106',r['as_of'])
checks['restored_model_prediction'] = np.allclose(restored['goals'],r['forecasts']['Argentina|France']['goals'])
old_view = RUN_DIR.name
migrate_workspace('/content/pitchops_workspace',WORKSPACE)
checks['drive_view_survives_migration'] = latest_dashboard(WORKSPACE)[1].name==old_view
print(json.dumps({'checks':checks,'checks_passed':sum(checks.values()),'research_run':r['run_id'],'dashboard':str(RUN_DIR/'dashboard.html')},ensure_ascii=False,indent=2))
if not all(checks.values()):
    raise ValueError('A persisted research invariant failed')


In [ ]:
from IPython.display import HTML, display
import html
from pitchops.storage import latest_dashboard
payload, RUN_DIR = latest_dashboard(WORKSPACE)
page = (RUN_DIR / 'dashboard.html').read_text()
display(HTML('<iframe title="PitchOps dashboard" style="width:100%;height:1100px;border:0;border-radius:16px" sandbox="allow-scripts allow-downloads" srcdoc="'+html.escape(page, quote=True)+'"></iframe>'))


## 12. Mở rộng dữ liệu có quyền truy cập (tùy chọn)

API-Football cần key do bạn cung cấp trong Colab Secrets với tên `API_FOOTBALL_KEY`. Kiểm tra coverage trước khi tải và tự chọn league/season; không tự mua gói, không tải hàng loạt toàn cầu. Adapter trả raw records có ID nguồn; cần mapping và kiểm định riêng trước khi đưa vào model đang dùng.


In [ ]:
CONNECT_LICENSED_PROVIDER = False
if CONNECT_LICENSED_PROVIDER:
    from google.colab import userdata
    from pitchops.providers import APIFootball
    provider = APIFootball(WORKSPACE / 'data' / 'api_football', userdata.get('API_FOOTBALL_KEY'), daily_budget=50)
    print(provider.coverage(league=39, season=2026))
else:
    print('Optional provider disabled. No API key needed for the open-data research pipeline.')


## 13. CI/CD hằng tuần và đồng bộ kết quả

GitHub Actions chạy thứ Sáu 14:23 giờ Việt Nam, có kiểm tra phục hồi thứ Bảy. Không cần mở Colab để job GitHub chạy. Chạy ô bên dưới để nhận model mới, giải mã bằng khóa trên Drive và giữ lại Event Lab/ảnh. [Cách vận hành và xem lỗi](https://github.com/HongQuang26/pitchops/blob/main/docs/CICD.md).


In [ ]:
# GitHub Actions performs the scheduled weekly collection/training.
# This cell only receives the latest approved model; it does not start extra downloads.
from pitchops.artifact_sync import sync_and_manage
print(sync_and_manage(WORKSPACE))
